In [10]:
import pandas as pd
import math

def split_unprocessed_data(input_file, chunk_size=10000):
   print(f"Loading {input_file}... ignoring the 'scraped_text' column.")
   
   # Explicitly list the columns we WANT to load, leaving out 'scraped_text'
   columns_to_keep = [
      'Business_Name', 'NAICS_Code', 'NAICS_Description', 'Insured Website', 
      'FULL_ADDRESS', 'inputs', 'sector_label', 'subsector_label', 
      'industry_label', 'summary'
   ]
   
   try:
      # Load the CSV using the python engine to bypass buffer limits
      df = pd.read_csv(
         input_file, 
         usecols=columns_to_keep, 
         encoding='latin-1', 
         engine='python',       # Swaps to the more robust parsing engine
         on_bad_lines='skip'    # Chucks the rows with unclosed quotes
      )
   except Exception as e:
      print(f"Failed to load CSV. Error: {e}")
      return

   # Ensure 'summary' exists to prevent KeyError
   if 'summary' not in df.columns:
      df['summary'] = None

   # FIND STARTING INDICES (Now only checking 'summary' since 'scraped_text' is gone)
   empty_mask = (df['summary'].isna() | (df['summary'].astype(str).str.strip() == ""))
   
   # Check if we actually have any rows matching the condition
   if not empty_mask.any():
      print("No rows found with an empty 'summary'. Exiting.")
      return

   # Find the index of the first occurrence
   start_index = empty_mask.idxmax()
   total_unprocessed = len(df) - start_index
   
   print(f"Found the first empty row at index {start_index}.")
   print(f"Total unprocessed rows to chunk: {total_unprocessed}")
   
   # Slice the dataframe starting from the first empty row
   df_unprocessed = df.iloc[start_index:]
   
   # Split and save into chunks
   num_chunks = math.ceil(len(df_unprocessed) / chunk_size)
   
   for i in range(num_chunks):
      # Calculate row boundaries for the current chunk
      start_row = i * chunk_size
      end_row = start_row + chunk_size
      
      # Extract the chunk
      chunk = df_unprocessed.iloc[start_row:end_row]
      
      # Save to a new CSV file (forcing utf-8 so the new files are clean)
      output_filename = f'unprocessed_data_part_{i+1}.csv'
      chunk.to_csv(output_filename, index=False, encoding='utf-8')
      print(f"Saved: {output_filename} ({len(chunk)} rows)")

   print("\nSplitting complete!")

# --- Execute the script ---
# Replace 'your_data.csv' with the actual name of your CSV file
split_unprocessed_data('website_summaries_train.csv')

Loading website_summaries_train.csv... ignoring the 'scraped_text' column.


Found the first empty row at index 22.
Total unprocessed rows to chunk: 76496
Saved: unprocessed_data_part_1.csv (10000 rows)
Saved: unprocessed_data_part_2.csv (10000 rows)
Saved: unprocessed_data_part_3.csv (10000 rows)
Saved: unprocessed_data_part_4.csv (10000 rows)
Saved: unprocessed_data_part_5.csv (10000 rows)
Saved: unprocessed_data_part_6.csv (10000 rows)
Saved: unprocessed_data_part_7.csv (10000 rows)
Saved: unprocessed_data_part_8.csv (6496 rows)

Splitting complete!
